# L05 answers（对照用）

对照 `../homework.md`。重点看：`fillna` 用列均值、`groupby`+`agg` 列名、`merge(how="left")` 与 `UNKNOWN` 城市。

## 练习 1 · 缺失处理后再求均值

In [ ]:
from pathlib import Path
import pandas as pd

# 本 notebook 在 L05/homework/ 下运行；CSV 在上一级 L05/
DATA = Path("..")

def filled_score_mean(csv_path: str) -> float:
    df = pd.read_csv(csv_path)
    # 填充前非缺失均值；fillna 后再对整列求均值
    m = df["score"].mean()  # 默认 skipna=True
    filled = df["score"].fillna(m)
    return float(filled.mean())

path = str(DATA / "scores.csv")
ans = filled_score_mean(path)
print(ans)
assert abs(ans - 68.8333333333) < 1e-6

# 小表自检：[10, None, 30] → 20.0
tiny = pd.DataFrame({"score": [10, None, 30]})
tiny_path = DATA / "_tiny_scores.csv"
tiny.to_csv(tiny_path, index=False)
assert filled_score_mean(str(tiny_path)) == 20.0
tiny_path.unlink(missing_ok=True)
print("ok: filled_score_mean")

## 练习 2 · groupby + agg

In [ ]:
def major_stats(df: pd.DataFrame) -> pd.DataFrame:
    clean = df.dropna(subset=["score"])
    out = (
        clean.groupby("major", as_index=False)
        .agg(
            avg_score=("score", "mean"),
            n=("score", "count"),
            total_credits=("credits", "sum"),
        )
    )
    return out[["major", "avg_score", "n", "total_credits"]]

scores = pd.read_csv(DATA / "scores.csv")
stats = major_stats(scores)
print(stats)

cs = stats.loc[stats["major"] == "计科"].iloc[0]
assert int(cs["n"]) == 2
assert abs(float(cs["avg_score"]) - 75.5) < 1e-9

fin = stats.loc[stats["major"] == "金融"].iloc[0]
assert int(fin["n"]) == 2
assert abs(float(fin["avg_score"]) - 58.5) < 1e-9
assert abs(float(fin["total_credits"]) - 4.0) < 1e-9  # 赵五 credits 缺失不计入

st = stats.loc[stats["major"] == "统计"].iloc[0]
assert int(st["n"]) == 2
assert abs(float(st["avg_score"]) - 72.5) < 1e-9
assert abs(float(st["total_credits"]) - 6.0) < 1e-9
print("ok: major_stats")

## 练习 3 · merge + 条件汇总

In [ ]:
def paid_amount_by_city(orders_csv: str, customers_csv: str) -> dict[str, float]:
    orders = pd.read_csv(orders_csv)
    customers = pd.read_csv(customers_csv)
    orders["amount"] = orders["amount"].fillna(0)
    merged = orders.merge(customers, on="customer_id", how="left")
    paid = merged[merged["status"] == "paid"].copy()
    paid["city"] = paid["city"].fillna("UNKNOWN")
    totals = paid.groupby("city")["amount"].sum()
    return {str(k): float(v) for k, v in totals.items()}

result = paid_amount_by_city(str(DATA / "orders.csv"), str(DATA / "customers.csv"))
print(result)
assert abs(result["上海"] - 120.5) < 1e-9
assert abs(result["北京"] - 80.0) < 1e-9
assert abs(result["UNKNOWN"] - 200.0) < 1e-9
print("ok: paid_amount_by_city")